In [1]:
%pwd

'c:\\Users\\HEMANGI\\Documents\\medical chatbot\\Medical-Chatbot-Generative-AI\\research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'c:\\Users\\HEMANGI\\Documents\\medical chatbot\\Medical-Chatbot-Generative-AI'

### Chunking operation

In [5]:
text_chunks = text_split(extracted_data)
print("length of text_chunks",len(text_chunks))

length of text_chunks 8759


In [6]:
# text_chunks

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
# downloading embedding from Hugging face
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [8]:
embeddings = download_hugging_face_embeddings()

In [9]:
query_result = embeddings.embed_query("Hello world")
print("length",len(query_result))

length 384


In [10]:
#query_result

### Initialize Pinecone


In [11]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
import os
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')



In [14]:
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "medicalbot"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,        # all-MiniLM-L6-v2
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

In [15]:
import os
os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["GROQ_API_KEY"]=GROQ_API_KEY

In [16]:
#Converts all your PDF text chunks into vector embeddings and stores them in the Pinecone vector database 
#so they can be quickly retrieved when users ask questions.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

In [17]:
from langchain_pinecone import PineconeVectorStore

# Load an existing Pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [18]:
docsearch

In [19]:
retriever = docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [20]:
retriever_docs = retriever.invoke("what is Metabolism ")

In [21]:
retriever_docs

[Document(id='a2e9f18d-fe84-4bd3-ad64-bf617bdcad1b', metadata={'creationdate': '2026-04-20T12:09:33-05:00', 'creator': 'PyPDF', 'moddate': '2026-04-20T17:26:55-05:00', 'page': 1145.0, 'page_label': '1146', 'producer': 'Prince 16.2 (www.princexml.com)', 'source': 'Data\\anatomy-and-physiology.pdf', 'title': 'Anatomy and Physiology 2e', 'total_pages': 1347.0}, page_content='Chapter Review \n24.1 Overview of Metabolic Reactions \nMetabolism is the sum of all catabolic (break down) \nand anabolic (synthesis) reactions in the body. The \nmetabolic rate measures the amount of energy used to \nmaintain life. An organism must ingest a sufficient \namount of food to maintain its metabolic rate if the \norganism is to stay alive for very long. \nCatabolic reactions break down larger molecules, such \nas carbohydrates, lipids, and proteins from ingested'),
 Document(id='cfae4953-de6d-4391-90ba-ec499db2cb8c', metadata={'creationdate': '2026-04-20T12:09:33-05:00', 'creator': 'PyPDF', 'moddate': '20

In [24]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # or another supported Groq model
    temperature=0.4,
    max_tokens=500,
    api_key=os.getenv("GROQ_API_KEY")
)

In [25]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise.\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [26]:
import langchain
import langchain_core

print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)

LangChain: 0.3.27
LangChain Core: 0.3.86


In [27]:
Question_ans_chain = create_stuff_documents_chain(llm,prompt)
Rag_chain = create_retrieval_chain(retriever,Question_ans_chain )

In [29]:
llm.invoke("Hello")

AIMessage(content='Hello. How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 36, 'total_tokens': 46, 'completion_time': 0.027437148, 'completion_tokens_details': None, 'prompt_time': 0.001067341, 'prompt_tokens_details': None, 'queue_time': 0.056102509, 'total_time': 0.028504489}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'finish_reason': 'stop', 'logprobs': None}, id='run--019fd7b5-e067-74e2-9141-12bd9adce75d-0', usage_metadata={'input_tokens': 36, 'output_tokens': 10, 'total_tokens': 46})

In [ ]:
text_chunks = text_split()

In [30]:
# Test 3
Question_ans_chain.invoke({
    "context": [],
    "input": "Hello"
})

"Hello, I'm here to help with any questions you may have. Please feel free to ask me anything, and I'll do my best to provide a helpful response. What's on your mind today?"

In [31]:
retriever.invoke("What is metabolism?")

[Document(id='a2e9f18d-fe84-4bd3-ad64-bf617bdcad1b', metadata={'creationdate': '2026-04-20T12:09:33-05:00', 'creator': 'PyPDF', 'moddate': '2026-04-20T17:26:55-05:00', 'page': 1145.0, 'page_label': '1146', 'producer': 'Prince 16.2 (www.princexml.com)', 'source': 'Data\\anatomy-and-physiology.pdf', 'title': 'Anatomy and Physiology 2e', 'total_pages': 1347.0}, page_content='Chapter Review \n24.1 Overview of Metabolic Reactions \nMetabolism is the sum of all catabolic (break down) \nand anabolic (synthesis) reactions in the body. The \nmetabolic rate measures the amount of energy used to \nmaintain life. An organism must ingest a sufficient \namount of food to maintain its metabolic rate if the \norganism is to stay alive for very long. \nCatabolic reactions break down larger molecules, such \nas carbohydrates, lipids, and proteins from ingested'),
 Document(id='1f54f9ed-b565-47ff-9e57-a25c0fdf3bdd', metadata={'creationdate': '2026-04-20T12:09:33-05:00', 'creator': 'PyPDF', 'moddate': '20

In [32]:
import traceback

try:
    response = Rag_chain.invoke({"input": "What is Metabolism?"})
    print(response["answer"])
except Exception:
    traceback.print_exc()

Metabolism is the sum of all catabolic (break down) and anabolic (synthesis) reactions in the body. It involves the breakdown of larger molecules, such as carbohydrates, lipids, and proteins, and the synthesis of new molecules. Essentially, metabolism is the process by which the body maintains life and energy.


In [33]:
response = Rag_chain.invoke({"input": "What is DNA?"})
print(response["answer"])

DNA (Deoxyribonucleic acid) is a nucleotide that stores genetic information. It contains deoxyribose, a phosphate group, and one nitrogen-containing base. The nitrogen-containing bases in DNA are adenine, cytosine, and others (though the text is cut off, typically also guanine and thymine).


In [34]:
response = Rag_chain.invoke({"input": "What is the heart?"})
print(response["answer"])

The heart can be thought of as a combination of two concepts: a pump and a muscle. It is a powerful engine that pumps blood throughout the body. The term "heart" is an English word, but cardiac terminology is rooted in the Greek term "kardia", which is studied by cardiologists.


In [37]:
response = Rag_chain.invoke({"input": "What is the geography?"})
print(response["answer"])

I don't know what the geography is, as the provided context only discusses anatomy, not geography. The context mentions specific body regions, such as the abdomen, but does not relate to geographical locations.
